# Dunnhumby 적응형 M2 단독 · 마지막 주 10시드

DAY1~704 학습, DAY705~711 test. seed42~51, 각300epoch 마지막 점만 평가합니다. M3/M4 없이 이진 그래프·plain BPR로 실행합니다. 중단되면 같은 실행 셀을 다시 눌러 저장된 epoch에서 재개하세요. 최종 성능 해석에는 동일 분할 M1 원본과 전체 지표 대조가 필요합니다.

In [ ]:
from pathlib import Path
from google.colab import drive
import json, subprocess, sys, torch

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요.'
SOURCE_COMMIT = '9891950363d9ea97e5d6a74df9176d12b6058d88'
REPO = Path('/content/clv-m2-adaptive-final-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','-q','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'fetch','-q','origin',SOURCE_COMMIT], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q','--detach',SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', torch.cuda.get_device_name(0), '| code:', SOURCE_COMMIT[:12])

In [ ]:
import clv_m2_adaptive_lastweek_final10 as screen
cfg = screen.configure('dunnhumby')
print(json.dumps(screen.preflight(cfg), ensure_ascii=False, indent=2))
prepared = screen.prepare(cfg)
print('분할:', prepared['data']['data_stats']['split_boundaries'])
print('데이터별 초기값:', prepared['protocol']['dataset_initialization'])

In [ ]:
result = screen.run_batch(prepared, cfg)
assert result['complete'] and len(result['completed_seeds']) == 10
print('10시드 종료. M1과의 검증된 비교 전까지 우열·유의성은 미판정입니다.')

In [ ]:
from IPython.display import display
display(result['summary'][result['summary'].metric.isin(['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50','price_purchase_amount_weighted_hit@10','vndcg@10'])])
print(result['paths'])

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
output = Path('/content/clv_m2_adaptive_dunnhumby_final10_results.zip')
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    archive.write(prepared['run_dir'] / 'protocol.json', 'protocol.json')
    for name, path in result['paths'].items():
        archive.write(path, Path(path).name)
files.download(str(output))